# One clock: crypto and the macro world in one DataFrame

Every TILLA series is labelled by the same UTC bar close, so crypto, stocks, FX and macro line up row for row with no
resampling, no timezone fixes and no forward-filling of your own. This notebook runs on the **free plan** (no card):

1. get a free key,
2. pull 30 days of hourly closes for every crypto and macro series the plan holds,
3. join them on the shared timestamp,
4. look at returns, correlation and a normalised chart.

Docs: https://tillacrypto.com/docs

In [ ]:
# pip install pandas matplotlib
import json, os, urllib.parse, urllib.request
from datetime import datetime, timedelta, timezone

import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://api.tillacrypto.com"
HEADERS = {"User-Agent": "tilla-notebook/1.0"}

## 1. A key

Set `TILLA_API_KEY` in your environment, or let the next cell ask the API for a free one
(one per address per day; sign up at https://tillacrypto.com for a key you can rotate).

In [ ]:
KEY = os.environ.get("TILLA_API_KEY", "").strip()
if not KEY:
    req = urllib.request.Request(BASE + "/v1/demo_key", data=b"", method="POST", headers=HEADERS)
    with urllib.request.urlopen(req, timeout=30) as r:
        answer = json.load(r)
    KEY = answer.get("key", "")
    if not KEY:
        raise SystemExit(answer.get("note", "No key returned - set TILLA_API_KEY and run again."))
TIER = "free"
print("key ready, plan:", TIER)

In [ ]:
def get(path, **params):
    q = urllib.parse.urlencode({k: v for k, v in params.items() if v is not None})
    req = urllib.request.Request(f"{BASE}{path}?{q}", headers={**HEADERS, "X-API-Key": KEY})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

def frame(lane, days, **params):
    """Every bar of a lane since `days` ago, following next_start until the window is complete."""
    start = (datetime.now(timezone.utc) - timedelta(days=days)).strftime("%Y-%m-%d %H:%M:%S")
    rows, cols, nxt = [], None, start
    while nxt:
        d = get(f"/v1/{TIER}/raw/{lane}", start=nxt, limit=5000, **params)
        cols = cols or ["time"] + d["series"]
        rows += d["data"]
        nxt = d["next_start"] if d["more"] else None
    df = pd.DataFrame(rows, columns=cols)
    df["time"] = pd.to_datetime(df["time"])
    return df.set_index("time")

## 2. Thirty days of hourly closes

`markets=` takes a whole category, so there is no list of names to maintain: the answer holds every series the
plan includes. Empty cells are real (a market was closed, or a source published nothing) - nothing is guessed.

In [ ]:
crypto = frame("hour", 30, markets="crypto", fields="close")
macro = frame("hour", 30, markets="macro", fields="close")
df = crypto.join(macro, how="outer")
print(df.shape, "->", list(df.columns))
df.tail()

## 3. One timestamp, every market

Both pulls came back on the same hourly clock, so the join is exact. Count how many bars carry a value per series:
24/7 crypto fills every hour; macro fills only the hours its market is open.

In [ ]:
coverage = df.notna().mean().sort_values(ascending=False).round(3)
coverage.to_frame("share of hours with a value")

## 4. Returns and correlation

Returns are taken per series on its own observed bars, then compared on the hours both have a value.

In [ ]:
ret = df.pct_change(fill_method=None)
corr = ret.corr(min_periods=48)
corr.round(2)

In [ ]:
norm = df.div(df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else 1))
ax = norm.plot(figsize=(11, 5), lw=1, legend=False, title="30 days, hourly, rebased to 1 (TILLA free plan)")
ax.set_xlabel("UTC")
plt.tight_layout()
plt.show()

## Where to go next

- **More markets and history:** paid plans add US stocks, FX, every BTC/ETH spot ETF fund, derivatives, the Bitcoin
  node and 293,743 indicators computed with no look-ahead - https://tillacrypto.com/pricing
- **Keep a local copy in sync:** `starter/tilla_dashboard.py --keep tilla.db` resumes missed bars and applies
  corrections.
- **Let an AI agent pull it:** `pip install tilla-mcp` (MCP server, 14 read-only tools).